In [1]:
from sklearn.metrics import mean_absolute_error
from keras.layers import Dense, Dropout
from keras.models import Sequential

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn import linear_model
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from math import sqrt
df = pd.read_excel("../../clean_coast_data.xlsx")
df.dropna(inplace=True)

In [2]:
# using get_dummies on education column
encoded = pd.get_dummies(df.Area)
# Concatenate the dummies to original dataframe
merged_data = pd.concat([df, encoded], axis=1)
# dropping the original column which was not encoded
merged_data.drop(columns='Area', inplace=True)

merged_data.drop(columns=['coast_name','coast_desc','city','check station','key','coast_num','center'], inplace=True)
merged_data['Date test num'] = merged_data['Date test'].view(np.int64)



In [3]:
X = merged_data.drop(columns=['test_result','test_result_num'])
y = merged_data[['test_result']]

In [4]:
X.drop('Date test',axis=1, inplace=True)
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=312148513)

X_train_scaler = MinMaxScaler()
X_test_scaler = MinMaxScaler()

X_train_scaled = X_train_scaler.fit_transform(X_train)
X_test_scaled = X_test_scaler.fit_transform(X_test)

# y_train_scaled = np.log(y_train+0.001)
# y_test_scaled = np.log(y_test+0.001)

dataset_train = pd.merge(left=X_train, right=y_train, left_index=True, right_index=True)
dataset_test = pd.merge(left=X_test, right=y_test, left_index=True, right_index=True)

In [7]:
from keras import backend
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard
from IPython.display import clear_output
from keras.regularizers import l2
from sklearn.metrics import r2_score

# create NN model
model = Sequential()
 
model.add(Dense(X_train.shape[1],input_dim=X_train.shape[1],activation='sigmoid'))

model.add(Dense(64,activation='sigmoid'))
model.add(Dropout(0.5))
model.add(Dense(32,activation='sigmoid'))
model.add(Dropout(0.5))
model.add(Dense(4,activation='sigmoid'))
model.add(Dropout(0.5))
model.add(Dense(1))
 
# Compiling the model
model.compile(metrics=['mae','accuracy'], optimizer='adam', loss = 'mean_squared_error')
es = EarlyStopping(monitor='val_loss', min_delta=1e-10, patience=25, verbose=1 )
# rlr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=0 )
# mcp = ModelCheckpoint(filepath='weights_Ramon.h5', monitor='val_loss', verbose=1 , save_best_only=True, save_weights_only=True)

# tb = TensorBoard('logs')

history = model.fit(X_train_scaled, y_train, shuffle=True, epochs=50 ,validation_split=0.3,batch_size=32, callbacks=[es])


def get_rmse(x,y):
  return sqrt(mean_squared_error(x,y))

#Train
prediction =model.predict(X_train_scaled)
print('')
resTrain = pd.DataFrame(
    data={
        'Prediction':prediction.T[0],
        'Actual': y_train.test_result.values
    },
    index=y_train.index
)

#Test
prediction =model.predict(X_test_scaled)
print('')
resTest = pd.DataFrame(
    data={
        'Prediction':prediction.T[0],
        'Actual': y_test.test_result.values
    },
    index=y_test.index
)

print('rmse train',sqrt(mean_squared_error(resTrain.Actual, resTrain.Prediction)))
print('std train',np.std(resTrain.Actual))
print('r2 train',r2_score(resTrain.Actual, resTrain.Prediction))
print('')

print('rmse test',sqrt(mean_squared_error(resTest.Actual, resTest.Prediction)))
print('std test',np.std(resTest.Actual))
print('r2 test',r2_score(resTest.Actual, resTest.Prediction))

Epoch 1/50
377/377 [==============================] - 1s 2ms/step - loss: 6.7784 - mae: 2.2438 - accuracy: 0.0178 - val_loss: 2.8157 - val_mae: 1.4828 - val_accuracy: 0.0207
Epoch 2/50
377/377 [==============================] - 1s 2ms/step - loss: 4.1726 - mae: 1.6669 - accuracy: 0.0209 - val_loss: 1.7186 - val_mae: 1.0925 - val_accuracy: 0.0207
Epoch 3/50
377/377 [==============================] - 1s 2ms/step - loss: 3.4851 - mae: 1.5216 - accuracy: 0.0236 - val_loss: 1.4735 - val_mae: 0.9940 - val_accuracy: 0.0207
Epoch 4/50
377/377 [==============================] - 1s 2ms/step - loss: 3.1214 - mae: 1.4379 - accuracy: 0.0236 - val_loss: 1.3709 - val_mae: 0.9735 - val_accuracy: 0.0207
Epoch 5/50
377/377 [==============================] - 1s 2ms/step - loss: 2.8188 - mae: 1.3688 - accuracy: 0.0236 - val_loss: 1.3070 - val_mae: 0.9593 - val_accuracy: 0.0207
Epoch 6/50
377/377 [==============================] - 1s 2ms/step - loss: 2.5008 - mae: 1.2837 - accuracy: 0.0236 - val_loss: 1.23